# VisionBridge — trained-model inference check (Colab)

This notebook is **inference/evaluation only**. The base model is already trained. It does **not** train, resume training, download the full dataset, or modify repository source code.

Goal: prove the trained checkpoint loads correctly and produces real text from a real keypoint sequence using the same model + decoder path used by VisionBridge inference.

Feature contract: pose=132, face=1404. Sequence length is capped by the existing repository collator/model limit of 1024 frames.


## 1. Install only missing inference dependencies

Do not uninstall or downgrade existing Colab packages. Do not reinstall PyTorch.


In [ ]:
import importlib.util, subprocess, sys

required = {
    'numpy': 'numpy==1.26.4',
    'google.protobuf': 'protobuf==4.25.9',
    'mediapipe': 'mediapipe==0.10.21',
    'cv2': 'opencv-python-headless',
    'pandas': 'pandas',
}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing only missing packages:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *missing], check=True)
    print('Installation finished. Restart the runtime once if MediaPipe/NumPy/Protobuf were newly installed.')
else:
    print('All required inference packages are already installed.')


## 2. Verify runtime


In [ ]:
import torch, numpy, google.protobuf, mediapipe as mp
print('NumPy:', numpy.__version__)
print('Protobuf:', google.protobuf.__version__)
print('MediaPipe:', mp.__version__)
print('Holistic API:', hasattr(mp, 'solutions'))
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert hasattr(mp, 'solutions'), 'MediaPipe Holistic is unavailable. Restart the runtime and rerun this cell.'


## 3. Load a fresh VisionBridge checkout

Only the notebook is being changed in Git. The backend/model source files are read-only for this test.


In [ ]:
import os, subprocess, sys
BASE='/content'
REPO_ROOT=f'{BASE}/VisionBridge'
if os.path.isdir(REPO_ROOT):
    subprocess.run(['git','-C',REPO_ROOT,'pull','--ff-only'], check=True)
else:
    subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',REPO_ROOT], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0,'backend')
print('Repository:', REPO_ROOT)

with open('backend/app/models/base_model.py', encoding='utf-8') as f:
    model_src=f.read()
with open('backend/scripts/extract_keypoints.py', encoding='utf-8') as f:
    extract_src=f.read()
assert 'POSE_INPUT_DIM = 33 * 4' in model_src and 'FACE_INPUT_DIM = 468 * 3' in model_src
assert 'POSE_FEATURE_DIM = 33 * 4' in extract_src and 'FACE_FEATURE_DIM = 468 * 3' in extract_src
assert '1434' not in model_src and '1434' not in extract_src
print('132/1404 feature contract verified.')


## 4. Locate the trained checkpoint + vocabulary

The notebook first checks the repository paths. If the trained model is not there, upload the `.pt` and `.vocab.json` files from your Colab/local machine. **No training happens here.**


In [ ]:
from pathlib import Path
import shutil

WEIGHTS=Path('backend/app/models/weights/base_model.pt')
VOCAB=Path('backend/app/models/weights/base_model.vocab.json')

if not WEIGHTS.exists() or not VOCAB.exists():
    print('Checkpoint/vocab not found in the checkout. Upload both files now.')
    from google.colab import files
    uploaded=files.upload()
    for name in ('base_model.pt','base_model.vocab.json'):
        if name not in uploaded:
            raise FileNotFoundError(f'Missing uploaded artifact: {name}')
        shutil.copy(name, 'backend/app/models/weights/'+name)
    print('Uploaded artifacts copied into backend/app/models/weights/.')

assert WEIGHTS.exists(), f'Missing weights: {WEIGHTS}'
assert VOCAB.exists(), f'Missing vocabulary: {VOCAB}'
print(f'Weights: {WEIGHTS} ({WEIGHTS.stat().st_size/1e6:.2f} MB)')
print(f'Vocab:   {VOCAB} ({VOCAB.stat().st_size/1e3:.2f} KB)')


## 5. Verify checkpoint integrity before inference

This catches wrong/corrupt checkpoints, vocabulary mismatches, and shape mismatches before any prediction is attempted.


In [ ]:
import torch
from app.models.base_model import load_frozen_base_model, POSE_INPUT_DIM, FACE_INPUT_DIM, MAX_SEQUENCE_LENGTH
from app.training.isltranslate import SimpleCharTokenizer

tokenizer=SimpleCharTokenizer.load(VOCAB)
state=torch.load(WEIGHTS, map_location='cpu')
assert isinstance(state, dict) and 'output_head.weight' in state, 'Checkpoint is not a VisionBridge state_dict with output_head.weight.'
checkpoint_vocab=int(state['output_head.weight'].shape[0])
print('Tokenizer vocabulary size:', tokenizer.vocab_size)
print('Checkpoint vocabulary size:', checkpoint_vocab)
assert checkpoint_vocab == tokenizer.vocab_size, 'Vocabulary size mismatch between checkpoint and vocab JSON.'

model=load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size)
param_count=sum(p.numel() for p in model.parameters())
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Model:', type(model).__name__)
print('Parameters:', f'{param_count:,}')
print('Trainable parameters:', f'{trainable:,}')
assert trainable == 0, 'Loaded base model is not frozen.'
print('CHECKPOINT INTEGRITY: PASS')


## 6. Test the model with an existing processed keypoint sample (preferred)

If `data/processed/isltranslate` is available, this is the cleanest check because it feeds the same 132/1404 representation used during training. No dataset download is performed.


In [ ]:
from pathlib import Path
import torch
from app.services.inference_service import decode_logits
from app.training.isltranslate import ISLTranslateKeypointDataset, _downsample_to_max_length

DATA_DIR=Path('data/processed/isltranslate')
has_processed=(DATA_DIR/'ISLTranslate.csv').exists() and (DATA_DIR/'pose').exists() and any((DATA_DIR/'pose').glob('*.npy'))

if has_processed:
    dataset=ISLTranslateKeypointDataset(DATA_DIR, tokenizer=tokenizer)
    print('Processed examples available:', len(dataset))
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model=load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size).to(device).eval()
    item=dataset[0]
    pose, face=_downsample_to_max_length(item['pose'], item['face'], item['uid'])
    print('UID:', item['uid'])
    print('Pose shape:', tuple(pose.shape))
    print('Face shape:', tuple(face.shape))
    assert pose.shape[-1] == POSE_INPUT_DIM == 132
    assert face.shape[-1] == FACE_INPUT_DIM == 1404
    assert pose.shape[0] <= MAX_SEQUENCE_LENGTH
    with torch.no_grad():
        logits=model(pose.unsqueeze(0).to(device), face.unsqueeze(0).to(device))
    prediction, confidence=decode_logits(logits)
    print('MODEL PREDICTION:', prediction)
    print('CONFIDENCE:', round(float(confidence),4))
    print('RAW LOGITS SHAPE:', tuple(logits.shape))
    print('MODEL INFERENCE CHECK: PASS')
else:
    print('No processed keypoint dataset found.')
    print('Use STEP 7 to test the trained model directly on a real webcam/video clip.')


## 7. Direct real-video inference check (no training data required)

Upload one short sign-language video. The notebook runs the repository's own MediaPipe extraction function, feeds the resulting 132/1404 sequence into the trained checkpoint, and prints the decoded prediction.


In [ ]:
from pathlib import Path
import shutil, numpy as np

test_dir=Path('data/model_check')
test_dir.mkdir(parents=True, exist_ok=True)

video_files=[]
try:
    from google.colab import files
    print('Upload one short sign-language video for model inference:')
    uploaded=files.upload()
    for name,data in uploaded.items():
        if name.lower().endswith(('.mp4','.avi','.mov','.mkv')):
            path=test_dir/name
            path.write_bytes(data)
            video_files.append(path)
except ImportError:
    pass

if video_files:
    from backend.scripts.extract_keypoints import extract_clip_keypoints
    with mp.solutions.holistic.Holistic(static_image_mode=False, model_complexity=1) as holistic:
        pose_np, face_np=extract_clip_keypoints(str(video_files[0]), holistic)
    print('Extracted pose:', pose_np.shape)
    print('Extracted face:', face_np.shape)
    assert pose_np.shape[1] == 132
    assert face_np.shape[1] == 1404

    pose=torch.from_numpy(pose_np).float()
    face=torch.from_numpy(face_np).float()
    pose,face=_downsample_to_max_length(pose,face,video_files[0].stem)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model=load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size).to(device).eval()
    with torch.no_grad():
        logits=model(pose.unsqueeze(0).to(device), face.unsqueeze(0).to(device))
    prediction,confidence=decode_logits(logits)
    print('\n==============================')
    print('VIDEO:', video_files[0].name)
    print('PREDICTED TEXT:', prediction)
    print('CONFIDENCE:', round(float(confidence),4))
    print('LATENT INPUT FRAMES:', pose.shape[0])
    print('==============================')
    print('REAL-VIDEO MODEL CHECK: PASS')
else:
    print('No video uploaded. You can skip this cell when a processed sample was already tested in STEP 6.')


## 8. Optional multi-sample health check — no training

Runs inference on up to 10 existing processed samples and reports how often the decoder returns non-empty text plus average confidence. It is a smoke test, not a benchmark.


In [ ]:
import statistics

if has_processed:
    n=min(10,len(dataset))
    confs=[]
    nonempty=0
    model=load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size).to(device).eval()
    with torch.no_grad():
        for i in range(n):
            item=dataset[i]
            pose,face=_downsample_to_max_length(item['pose'],item['face'],item['uid'])
            logits=model(pose.unsqueeze(0).to(device),face.unsqueeze(0).to(device))
            pred,conf=decode_logits(logits)
            confs.append(float(conf))
            if pred and pred != '(no sign detected)':
                nonempty += 1
            print(f'{i+1:02d}. {item["uid"]}: {pred!r}  conf={float(conf):.3f}')
    print('\nSamples checked:', n)
    print('Non-empty predictions:', f'{nonempty}/{n}')
    print('Average confidence:', round(statistics.mean(confs),4))
    print('MULTI-SAMPLE SMOKE TEST: PASS')
else:
    print('Skipped — no processed dataset available.')


## Result

A successful run means: the checkpoint loads, vocabulary matches, the base model is frozen, the 132/1404 input contract is correct, the decoder executes, and at least one real input produces a model prediction.

This notebook intentionally does **not** train or claim that one smoke-test prediction proves model accuracy. Use a labeled evaluation dataset separately when you need CER/WER metrics.
